# 01 - Data inventory and construction of the TfL commuter-demand shock

This notebook establishes the data-entry structure for the full dissertation workflow and reconstructs the station-level commuter-demand measure used in Objective 1 directly from the four public TfL annualised entry/exit files. The Top-100 station table is therefore an **output of this notebook, not an external input**.

The computational workflow in this notebook:

1. audits where each public, external and safeguarded dataset first enters the analysis;
2. harmonises station names and weekday columns across 2019, 2023, 2024 and 2025;
3. combines entries and exits for Monday, midweek and Friday;
4. measures how Monday and Friday activity changed relative to the available 2019 working-week pattern;
5. sums the 2023-2025 station scores and ranks the Top 100;
6. attaches public TfL station-point geometry and writes reusable derived files.

The earlier project-development file `Top100_Master_Spatial_OD_Sheet_Clean.csv` is not required. Its LAD and Top-3-origin columns belonged to an earlier screening approach. The final dissertation derives residential origins later from MSOA-to-MSOA Census commuting flows.


In [ ]:
from pathlib import Path
import os
import re

import numpy as np
import pandas as pd
import geopandas as gpd


def repository_root():
    configured = os.environ.get("DISSERTATION_WORKSPACE")
    if configured:
        candidate = Path(configured).expanduser().resolve()
        if (candidate / "data" / "public").exists():
            return candidate
    here = Path.cwd().resolve()
    if here.name == "notebooks" and (here.parent / "data" / "public").exists():
        return here.parent
    if (here / "data" / "public").exists():
        return here
    raise FileNotFoundError("Run this notebook from the repository root or notebooks directory.")


ROOT = repository_root()
TFL_DIR = ROOT / "data" / "public" / "tfl"
BOUNDARY_DIR = ROOT / "data" / "public" / "boundaries"
DERIVED_DIR = ROOT / "data" / "derived"
DERIVED_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    "2019": TFL_DIR / "AnnualisedEntryExit_2019.csv",
    "2023": TFL_DIR / "AC2023_AnnualisedEntryExit.csv",
    "2024": TFL_DIR / "AC2024_AnnualisedEntryExit_Public.csv",
    "2025": TFL_DIR / "AC2025_AnnualisedEntryExit_public.csv",
    "station_points": BOUNDARY_DIR / "Underground_Stations.geojson",
}

missing = [name for name, path in FILES.items() if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing public inputs: {missing}")

print("Repository:", ROOT)
for name, path in FILES.items():
    print(f"{name}: {path.relative_to(ROOT)}")


## Data inputs and their entry points

Notebook 01 constructs only the TfL station measure because it can be reproduced from the public files included in this repository. The remaining datasets have not disappeared: they enter the workflow in the notebooks shown below. Safeguarded Green Street inputs are listed for transparency but are read only inside an authorised local environment. The complete ONS MSOA commuting matrix is included in compressed form. OpenLocal study-scope analysis tables are included, while the much larger property-level source is obtained from the official service when record-level reconstruction is required.

This separation prevents a derived Top-100 table from being mistaken for source data while keeping the complete analytical sequence visible.


In [ ]:
data_entry = pd.DataFrame([
    {
        "Dataset": "TfL annualised station entry/exit data",
        "Access": "Public; included",
        "First notebook": "01",
        "Role": "Construct commuter-demand shock and Top-100 stations",
    },
    {
        "Dataset": "TfL Underground station point geometry",
        "Access": "Public; included",
        "First notebook": "01",
        "Role": "Attach coordinates to ranked stations",
    },
    {
        "Dataset": "Green Street historical tenant-premises POI data",
        "Access": "Commercially safeguarded; not distributed",
        "First notebook": "02 and 08",
        "Role": "Construct active stock, openings, closures, turnover and net formation",
    },
    {
        "Dataset": "Green Street aggregate vacancy extracts",
        "Access": "Commercially safeguarded; not distributed",
        "First notebook": "03 and 06",
        "Role": "Construct consumer-facing vacancy and long-term vacancy measures",
    },
    {
        "Dataset": "OpenLocal commercial-property data",
        "Access": "Openly licensed; study-scope analysis tables included",
        "First notebook": "03; then 05, 06, 09 and 10",
        "Role": "Construct retail and office stock, floorspace, value and occupation measures",
    },
    {
        "Dataset": "ONS 2021 Census origin-destination commuting flows",
        "Access": "Public; complete compressed matrix included",
        "First notebook": "05",
        "Role": "Link workplace MSOAs to residential-origin MSOAs and calculate exposure",
    },
    {
        "Dataset": "London office-market and statistical boundaries",
        "Access": "Public; office-market, LAD, MSOA and station layers included",
        "First notebook": "04",
        "Role": "Define the five office submarkets and aggregate records to analysis geographies",
    },
])

display(data_entry)


## 1. Harmonise the four TfL files

TfL changed its weekday column labels between releases. In 2019 the first two weekday groups used here are Monday-Thursday and Friday. From 2023 onward, Monday, Tuesday-Thursday and Friday are available separately. Entries and exits are added so that each value represents total station activity for that day group.

Station names are normalised only to reconcile formatting differences such as `&` versus `and`, punctuation in `St.`, and mode suffixes. This does not alter the underlying passenger counts.


In [ ]:
def clean_station_name(value):
    if not isinstance(value, str):
        return ""
    name = value.strip()
    name = re.sub(r"\s+\((?:LU|LO|DLR|NR)\)$", "", name, flags=re.IGNORECASE)
    name = re.sub(r"\s+(?:LU|LO|DLR|NR)$", "", name, flags=re.IGNORECASE)
    name = name.replace(" & ", " and ").replace("St. ", "St ")
    name = re.sub(r"\s+", " ", name)
    return name.strip()


def numeric(series):
    return pd.to_numeric(series.astype(str).str.replace(",", "", regex=False), errors="coerce")


def load_2019(path):
    frame = pd.read_csv(path, skiprows=6).dropna(subset=["Station"]).copy()
    for column in ["entries", "entries.1", "exits", "exits.1"]:
        frame[column] = numeric(frame[column])
    frame["clean_name"] = frame["Station"].map(clean_station_name)
    frame["2019_Midweek"] = frame["entries"] + frame["exits"]
    frame["2019_Friday"] = frame["entries.1"] + frame["exits.1"]
    return frame.loc[frame["Mode"].eq("LU"), ["clean_name", "2019_Midweek", "2019_Friday"]]


def load_2023(path):
    frame = pd.read_csv(path, skiprows=6).dropna(subset=["Station"]).copy()
    columns = ["Entries", "Entries.1", "Entries.2", "Exits", "Exits.1", "Exits.2"]
    for column in columns:
        frame[column] = numeric(frame[column])
    frame["clean_name"] = frame["Station"].map(clean_station_name)
    frame["2023_Monday"] = frame["Entries"] + frame["Exits"]
    frame["2023_Midweek"] = frame["Entries.1"] + frame["Exits.1"]
    frame["2023_Friday"] = frame["Entries.2"] + frame["Exits.2"]
    return frame.loc[frame["Mode"].eq("LU"), ["clean_name", "2023_Monday", "2023_Midweek", "2023_Friday"]]


def load_2024_2025(path, year):
    frame = pd.read_csv(path, skiprows=5).iloc[1:].dropna(subset=["Station"]).copy()
    columns = ["Monday", "Midweek (Tue-Thu)", "Friday", "Monday.1", "Midweek (Tue-Thu).1", "Friday.1"]
    for column in columns:
        frame[column] = numeric(frame[column])
    frame["clean_name"] = frame["Station"].map(clean_station_name)
    frame[f"{year}_Monday"] = frame["Monday"] + frame["Monday.1"]
    frame[f"{year}_Midweek"] = frame["Midweek (Tue-Thu)"] + frame["Midweek (Tue-Thu).1"]
    frame[f"{year}_Friday"] = frame["Friday"] + frame["Friday.1"]
    keep = ["clean_name", f"{year}_Monday", f"{year}_Midweek", f"{year}_Friday"]
    return frame.loc[frame["Mode"].eq("LU"), keep]


station_years = {
    "2019": load_2019(FILES["2019"]),
    "2023": load_2023(FILES["2023"]),
    "2024": load_2024_2025(FILES["2024"], 2024),
    "2025": load_2024_2025(FILES["2025"], 2025),
}

for year, frame in station_years.items():
    duplicates = frame["clean_name"].duplicated().sum()
    print(f"{year}: {len(frame)} LU records; duplicate cleaned names={duplicates}")


## 2. Construct the commuter-demand shock score

For each post-pandemic year, Monday and Friday are first expressed relative to the Tuesday-Thursday midweek level. The available 2019 data combine Monday-Thursday, so this grouped weekday level is used as the Monday reference and assigned a ratio of 1. Friday is separately normalised against the observed 2019 Friday-to-midweek ratio.

For station $s$, year $t$ and day group $d$, the deficit is:

$$Deficit_{s,t,d}=1-\frac{Ratio_{s,t,d}}{Ratio_{s,2019,d}}$$

The annual score adds the Monday and Friday deficits. The final commuter-demand shock score sums the annual scores for 2023, 2024 and 2025. A larger positive score means that the station's Monday/Friday pattern weakened more strongly relative to its 2019 reference; it is not a direct measure of remote-working prevalence or total economic decline.


In [ ]:
station_panel = station_years["2019"]
for year in ["2023", "2024", "2025"]:
    station_panel = station_panel.merge(station_years[year], on="clean_name", how="inner", validate="one_to_one")

station_panel["raw_ratio_19_mon"] = 1.0
station_panel["raw_ratio_19_fri"] = station_panel["2019_Friday"] / station_panel["2019_Midweek"].replace(0, np.nan)

for short, full in [("23", "2023"), ("24", "2024"), ("25", "2025")]:
    station_panel[f"raw_ratio_{short}_mon"] = (
        station_panel[f"{full}_Monday"] / station_panel[f"{full}_Midweek"].replace(0, np.nan)
    )
    station_panel[f"raw_ratio_{short}_fri"] = (
        station_panel[f"{full}_Friday"] / station_panel[f"{full}_Midweek"].replace(0, np.nan)
    )
    station_panel[f"collapse_{short}_mon"] = 1 - station_panel[f"raw_ratio_{short}_mon"]
    station_panel[f"collapse_{short}_fri"] = 1 - (
        station_panel[f"raw_ratio_{short}_fri"] / station_panel["raw_ratio_19_fri"]
    )
    station_panel[f"total_collapse_{short}"] = (
        station_panel[f"collapse_{short}_mon"] + station_panel[f"collapse_{short}_fri"]
    )

station_panel["cumulative_collapse_score"] = station_panel[
    ["total_collapse_23", "total_collapse_24", "total_collapse_25"]
].sum(axis=1, min_count=3)
station_panel = station_panel.replace([np.inf, -np.inf], np.nan)

required = ["cumulative_collapse_score", "2019_Midweek"]
valid = station_panel.dropna(subset=required).copy()
valid = valid.sort_values("cumulative_collapse_score", ascending=False).reset_index(drop=True)
valid["shock_rank"] = np.arange(1, len(valid) + 1)
valid["top100_affected"] = valid["shock_rank"].le(100)

top100 = valid.loc[valid["top100_affected"]].copy()
print(f"Stations observed in all four years with valid scores: {len(valid)}")
print(f"Top-100 records: {len(top100)}")
display(top100[["shock_rank", "clean_name", "cumulative_collapse_score"]].head(10))


## 3. Attach public station geometry

Station-point geometry is joined by the same cleaned station name. TfL's spatial layer represents Bank and Monument separately, while the passenger table uses the combined name `Bank and Monument`; the Bank point is used for that combined record. Any remaining unmatched names are reported rather than silently discarded.


In [ ]:
points = gpd.read_file(FILES["station_points"]).to_crs("EPSG:4326")
points["clean_name"] = points["NAME"].map(clean_station_name)
points = points[["clean_name", "geometry"]].drop_duplicates("clean_name")

if "Bank" in set(points["clean_name"]):
    bank_geometry = points.loc[points["clean_name"].eq("Bank"), "geometry"].iloc[0]
    points = pd.concat(
        [points, gpd.GeoDataFrame({"clean_name": ["Bank and Monument"]}, geometry=[bank_geometry], crs="EPSG:4326")],
        ignore_index=True,
    )

# Explicit aliases reconcile line-specific labels used by the passenger and
# spatial releases. They change names only; passenger values remain untouched.
point_lookup = points.set_index("clean_name")["geometry"]
station_aliases = {
    "Paddington TfL": "Paddington",
    "Hammersmith (DIS)": "Hammersmith (Dist&Picc Line)",
    "Hammersmith (H&C)": "Hammersmith (H&C Line)",
    "Edgware Road (Bak)": "Edgware Road (Bakerloo)",
    "Edgware Road (DIS)": "Edgware Road (Circle Line)",
    "Heathrow Terminals 123": "Heathrow Terminals 1-2-3",
}
alias_rows = [
    {"clean_name": passenger_name, "geometry": point_lookup[point_name]}
    for passenger_name, point_name in station_aliases.items()
    if point_name in point_lookup.index
]
if alias_rows:
    points = pd.concat(
        [points, gpd.GeoDataFrame(alias_rows, geometry="geometry", crs="EPSG:4326")],
        ignore_index=True,
    )

station_spatial = valid.merge(points, on="clean_name", how="left", validate="one_to_one")
unmatched = station_spatial.loc[station_spatial["geometry"].isna(), "clean_name"].tolist()
print("Unmatched station names:", unmatched if unmatched else "none")

station_spatial = gpd.GeoDataFrame(station_spatial, geometry="geometry", crs="EPSG:4326")
top100_spatial = station_spatial.loc[station_spatial["top100_affected"]].copy()
if len(top100_spatial) != 100 or top100_spatial["geometry"].isna().any():
    raise ValueError("Top-100 station geometry matching is incomplete.")


## 4. Write derived outputs

The CSV files retain geometry as WKT for compatibility with the downstream notebooks. GeoJSON versions are also supplied for direct inspection in GIS software. These files are derived entirely from public TfL inputs and can be regenerated by running this notebook.


In [ ]:
def csv_with_wkt(frame, path):
    table = pd.DataFrame(frame.drop(columns="geometry"))
    table["geometry"] = frame.geometry.map(lambda geom: geom.wkt if geom is not None else None)
    table.to_csv(path, index=False)


all_csv = DERIVED_DIR / "tfl_all_station_commuter_shock.csv"
top100_csv = DERIVED_DIR / "tfl_top100_commuter_shock_stations.csv"
all_geojson = DERIVED_DIR / "tfl_all_station_commuter_shock.geojson"
top100_geojson = DERIVED_DIR / "tfl_top100_commuter_shock_stations.geojson"

csv_with_wkt(station_spatial, all_csv)
csv_with_wkt(top100_spatial, top100_csv)
station_spatial.to_file(all_geojson, driver="GeoJSON")
top100_spatial.to_file(top100_geojson, driver="GeoJSON")

for path in [all_csv, top100_csv, all_geojson, top100_geojson]:
    print(path.relative_to(ROOT))
